### Load System Libraries

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join("..", "useful-python-scripts-eda")))
from data_profiler import DataProfiler
from distribution_analyzer import DistributionAnalyzer
from correlation_explorer import CorrelationExplorer
from outlier_suite import OutlierSuite
from missing_data_analyzer import MissingDataAnalyzer

import random
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

import kaggle

### Load Datasets

In [ ]:
# Define input folder.
input = "input/berka-dataset"

In [ ]:
# Authenticate using kaggle.json (in .kaggle directory at home)
kaggle.api.authenticate()

In [ ]:
# Create input and output folders if they don't exist.
os.makedirs(input, exist_ok=True)

In [ ]:
# Download Kaggle Berka Dataset.
kaggle.api.dataset_download_files(
    "marceloventura/the-berka-dataset",
    path=input,
    unzip=True,
    quiet=False,
)

In [ ]:
# Load the dataset.
account_df = pd.read_csv(os.path.join(input, "account.csv"), sep=";")
card_df = pd.read_csv(os.path.join(input, "card.csv"), sep=";")
client_df = pd.read_csv(os.path.join(input, "client.csv"), sep=";")
disp_df = pd.read_csv(os.path.join(input, "disp.csv"), sep=";")
dist_df = pd.read_csv(os.path.join(input, "district.csv"), sep=";")
loan_df = pd.read_csv(os.path.join(input, "loan.csv"), sep=";")
order_df = pd.read_csv(os.path.join(input, "order.csv"), sep=";")
trans_df = pd.read_csv(os.path.join(input, "trans.csv"), sep=";", low_memory=False)

### Generate master database

In [ ]:
# Some feature engineering for better readability and easier analysis.
account_df["date"] = pd.to_datetime(account_df["date"], format="%y%m%d")
account_df = account_df.rename(columns={"date":"account_open_date"})

account_df["frequency"] = account_df["frequency"].replace(
    {"POPLATEK MESICNE":"Monthly Issuance",
     "POPLATEK TYDNE":"Weekly Issuance",
     "POPLATEK PO OBRATU":"Issuance After Transaction"}
    )

dist_df = dist_df.rename(columns={"A1":"district_id", 
    "A3":"region_name",
    "A12":"unemployment_rate_95",
    "A15":"n_crimes_95"}
    )

loan_df["date"] = pd.to_datetime(loan_df["date"], format="%y%m%d")

In [ ]:
# For this example we will focus on the loans dataset, and join it with accounts dataset.
account_cols = ["account_id", "district_id", "frequency", "account_open_date"]
district_cols = ["district_id", "region_name", "unemployment_rate_95", "n_crimes_95"]


df = loan_df.merge(account_df[account_cols], on="account_id", how="left")
df = df.merge(dist_df[district_cols], on="district_id", how="left")

In [ ]:
# Some columns are clearly categorical, so we transform them to string datatype.
categorical_cols = ["loan_id", "account_id", "district_id"]
for col in categorical_cols:
    df[col] = df[col].astype(str)

In [ ]:
# Calculate tenure (account age) from the date column with respect to the account opening date.
df["account_tenure"] = (df["date"] - df["account_open_date"]).dt.days
df["account_tenure"] = df["account_tenure"] / 30.44
df["account_tenure"] = df["account_tenure"].round().astype(int)
df = df.drop(columns=["account_open_date"])

In [ ]:
for col in ["unemployment_rate_95", "n_crimes_95"]:
    frac = random.uniform(0.01, 0.02)
    idx = df.sample(frac=frac, random_state=42 if col == "unemployment_rate_95" else 99).index
    df.loc[idx, col] = pd.NA
    print(f"Introduced nulls in {col}: {len(idx)} rows ({len(idx)/len(df):.3%})")

In [ ]:
df.head(5)

### Comprehensive Data Profiler

In [ ]:
# Create profiler.
profiler = DataProfiler(df, high_cardinality_threshold=0.5)

In [ ]:
# Print summary to console.
profiler.print_summary()

In [ ]:
# Get detailed report.
profiler_report = profiler.generate_full_profile()
print(profiler_report["overview"])
print("--" * 20)
print(profiler_report["numeric_profiles"])
print("--" * 20)
print(profiler_report["categorical_profiles"])
print("--" * 20)
print(profiler_report["data_quality_issues"])

### Analyzing And Visualizing Distributions

In [ ]:
analyzer = DistributionAnalyzer(df)

In [ ]:
# Generate distribution report
analyzer_report = analyzer.generate_distribution_report()
print(analyzer_report)

In [ ]:
# Identify highly skewed columns
skewed = analyzer_report[abs(analyzer_report['skewness']) > 2]
print(f"Highly skewed columns: {skewed['column'].tolist()}")

In [ ]:
# Visualize distributions
analyzer.plot_numeric_distributions(max_cols=10)
analyzer.plot_boxplots()
analyzer.plot_qq_plots()
analyzer.plot_categorical_distributions()

### Correlation and Relationship Explorer

In [ ]:
explorer = CorrelationExplorer(df)

In [ ]:
# Find high correlations
high_corr = explorer.find_high_correlations(threshold=0.7, method='pearson')
print(high_corr)

In [ ]:
# Check for multicollinearity
vif = explorer.calculate_vif()
problematic = vif[vif['vif'] > 10]
print(f"Features with high multicollinearity:\n{problematic}")

In [ ]:
# Mutual information with target
if 'target' in df.columns:
    mi_scores = explorer.mutual_information_analysis('target')
    print(f"Top features:\n{mi_scores.head(10)}")

In [ ]:
# Visualize
explorer.plot_correlation_heatmap(method='pearson')
explorer.plot_correlation_comparison()
explorer.plot_scatter_matrix(max_cols=5)
explorer.plot_top_correlations(n_pairs=10)

### Outlier Detection and Analysis Suite

In [ ]:
suite = OutlierSuite(df)

In [ ]:
# Compare methods across all columns
summary = suite.compare_methods_all_columns()
print(summary)

In [ ]:
# Analyze specific column
col = "amount"
suite.plot_outlier_comparison(col)

In [ ]:
# Detect multivariate outliers
iso_outliers = suite.detect_isolation_forest_outliers(contamination=0.1)
print(f"Found {iso_outliers.sum()} multivariate outliers")

feature1 = "amount"
feature2 = "account_tenure"
suite.plot_multivariate_outliers([feature1, feature2])

In [ ]:
# Analyze outlier impact
col = "amount"
impact = suite.analyze_outlier_impact(col)
print(impact)

### Missing Data Pattern Analyzer

In [ ]:
analyzer = MissingDataAnalyzer(df)

In [ ]:
# Generate full report
report = analyzer.generate_full_report()

print("Missing Value Summary:")
print(report['summary'])

print("\nMissingness Patterns (co-occurrence):")
print(report['patterns'])

print("\nMissingness Classifications:")
print(report['classifications'])

print("\nImputation Recommendations:")
print(report['recommendations'])

In [ ]:
# Visualize patterns
analyzer.plot_missing_bar()
analyzer.plot_missing_heatmap(max_cols=30)
analyzer.plot_missing_correlation()

In [ ]:
# Classify specific column
col = "n_crimes_95"
classification = analyzer.classify_missingness_type(col)
recommendation = analyzer.recommend_strategy(col)
print(f"Missingness type: {classification['missingness_type']}")
print(f"Recommendation: {recommendation}")